In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
VISUALISASI PERBANDINGAN HASIL BENCHMARKING MCU-QUAKE 1C vs 3C
Data Indonesia - Berdasarkan output numerik yang diberikan.
"""

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
from pathlib import Path

# =============================================
# 1. DATA DARI HASIL BENCHMARKING
# =============================================

# Confusion Matrix (sama untuk 1C dan 3C)
# Berdasarkan angka: NO (TP=7453, FP=5486), LE (FN=2414, TN=10525)
cm = np.array([
    [7453, 5486],   # NO: TP, FP
    [2414, 10525]   # LE: FN, TN
])

# Metrik 1C (dari output Anda)
metrics_1C = {
    'accuracy': 0.6947213849601979,
    'precision': [0.7553461031721901, 0.6573605646118293],  # NO, LE
    'recall': [0.5760105108586444, 0.8134322590617513],     # NO, LE
    'f1': [0.6535999298430237, 0.7271157167530224],         # NO, LE
    'specificity': [0.8134322590617513, 0.5760105108586444] # NO, LE
}

# Metrik 3C (dari output Anda)
metrics_3C = {
    'accuracy': 0.6925903008740042,
    'precision': [0.7395612853569367, 0.6610191412312467],
    'recall': [0.5945548766339237, 0.7906257251140846],
    'f1': [0.6591776358101444, 0.720036628746522],
    'specificity': [0.7906257251140846, 0.5945548766339237]
}

# Label
labels = ["NO (Noise)", "LE (Gempa)"]
OUTPUT_DIR = "/Volumes/Extreme SSD/mcu_quake_output_replikasi_demo/indonesia_domain/visualisasi_perbandingan"
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

# =============================================
# 2. PLOT CONFUSION MATRIX (1C dan 3C - SAMA)
# =============================================

def plot_confusion_matrices():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Normalisasi per baris
    cm_percent = cm / cm.sum(axis=1, keepdims=True) * 100
    
    # 1C
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax1,
                cbar=False, annot_kws={'size': 14})
    ax1.set_title('MCU-Quake 1C (Z only)', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Predicted Label', fontsize=12)
    ax1.set_ylabel('True Label', fontsize=12)
    ax1.text(0.5, -0.15, f"Accuracy: {metrics_1C['accuracy']*100:.2f}%", 
             transform=ax1.transAxes, ha='center', fontsize=12)
    
    # 3C
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=labels, yticklabels=labels, ax=ax2,
                cbar=False, annot_kws={'size': 14})
    ax2.set_title('MCU-Quake 3C (Z+N+E)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Predicted Label', fontsize=12)
    ax2.set_ylabel('True Label', fontsize=12)
    ax2.text(0.5, -0.15, f"Accuracy: {metrics_3C['accuracy']*100:.2f}%",
             transform=ax2.transAxes, ha='center', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix_1C_vs_3C.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Confusion matrix saved to {OUTPUT_DIR}/confusion_matrix_1C_vs_3C.png")

# =============================================
# 3. PLOT PERBANDINGAN METRIK
# =============================================

def plot_metrics_comparison():
    # Siapkan data
    metrics_names = ['Accuracy', 'Precision NO', 'Precision LE', 
                     'Recall NO', 'Recall LE', 'F1 NO', 'F1 LE']
    
    values_1C = [
        metrics_1C['accuracy']*100,
        metrics_1C['precision'][0]*100,
        metrics_1C['precision'][1]*100,
        metrics_1C['recall'][0]*100,
        metrics_1C['recall'][1]*100,
        metrics_1C['f1'][0]*100,
        metrics_1C['f1'][1]*100
    ]
    
    values_3C = [
        metrics_3C['accuracy']*100,
        metrics_3C['precision'][0]*100,
        metrics_3C['precision'][1]*100,
        metrics_3C['recall'][0]*100,
        metrics_3C['recall'][1]*100,
        metrics_3C['f1'][0]*100,
        metrics_3C['f1'][1]*100
    ]
    
    df = pd.DataFrame({
        'Metric': metrics_names,
        '1C': values_1C,
        '3C': values_3C
    })
    
    fig, ax = plt.subplots(figsize=(12, 7))
    x = np.arange(len(df['Metric']))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, df['1C'], width, label='1C (Z only)', color='steelblue', alpha=0.8)
    bars2 = ax.bar(x + width/2, df['3C'], width, label='3C (Z+N+E)', color='coral', alpha=0.8)
    
    ax.set_xlabel('Metric', fontsize=12)
    ax.set_ylabel('Percentage (%)', fontsize=12)
    ax.set_title('Perbandingan Metrik MCU-Quake: 1C vs 3C (Data Indonesia)', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(df['Metric'], rotation=15, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 100)
    
    # Tambahkan nilai di atas bar
    for bar in bars1:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{height:.1f}', ha='center', va='bottom', fontsize=8)
    for bar in bars2:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{height:.1f}', ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'metrics_comparison.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Metrics comparison saved to {OUTPUT_DIR}/metrics_comparison.png")
    
    # Simpan CSV
    df.to_csv(os.path.join(OUTPUT_DIR, 'metrics_comparison.csv'), index=False)
    print(f"✅ Metrics CSV saved to {OUTPUT_DIR}/metrics_comparison.csv")

# =============================================
# 4. PLOT SPECIFICITY (TNR)
# =============================================

def plot_specificity():
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(labels))
    width = 0.35
    
    spec_1C = [metrics_1C['specificity'][0]*100, metrics_1C['specificity'][1]*100]
    spec_3C = [metrics_3C['specificity'][0]*100, metrics_3C['specificity'][1]*100]
    
    ax.bar(x - width/2, spec_1C, width, label='1C (Z only)', color='steelblue')
    ax.bar(x + width/2, spec_3C, width, label='3C (Z+N+E)', color='coral')
    
    ax.set_xlabel('Class', fontsize=12)
    ax.set_ylabel('Specificity (TNR) (%)', fontsize=12)
    ax.set_title('Perbandingan Specificity (True Negative Rate)', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 100)
    
    # Tambahkan nilai
    for i, (v1, v2) in enumerate(zip(spec_1C, spec_3C)):
        ax.text(i - width/2, v1 + 0.5, f'{v1:.1f}%', ha='center', va='bottom', fontsize=10)
        ax.text(i + width/2, v2 + 0.5, f'{v2:.1f}%', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'specificity_comparison.png'), dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Specificity comparison saved to {OUTPUT_DIR}/specificity_comparison.png")

# =============================================
# 5. RINGKASAN TABEL
# =============================================

def print_summary():
    print("\n" + "="*60)
    print("📊 RINGKASAN METRIK MCU-QUAKE INDONESIA")
    print("="*60)
    print(f"{'Metric':<20} {'1C':<12} {'3C':<12} {'Delta':<10}")
    print("-"*60)
    print(f"{'Accuracy':<20} {metrics_1C['accuracy']*100:.2f}%    {metrics_3C['accuracy']*100:.2f}%    {(metrics_3C['accuracy']-metrics_1C['accuracy'])*100:+.2f}%")
    print(f"{'Precision NO':<20} {metrics_1C['precision'][0]*100:.2f}%    {metrics_3C['precision'][0]*100:.2f}%    {(metrics_3C['precision'][0]-metrics_1C['precision'][0])*100:+.2f}%")
    print(f"{'Precision LE':<20} {metrics_1C['precision'][1]*100:.2f}%    {metrics_3C['precision'][1]*100:.2f}%    {(metrics_3C['precision'][1]-metrics_1C['precision'][1])*100:+.2f}%")
    print(f"{'Recall NO':<20} {metrics_1C['recall'][0]*100:.2f}%    {metrics_3C['recall'][0]*100:.2f}%    {(metrics_3C['recall'][0]-metrics_1C['recall'][0])*100:+.2f}%")
    print(f"{'Recall LE':<20} {metrics_1C['recall'][1]*100:.2f}%    {metrics_3C['recall'][1]*100:.2f}%    {(metrics_3C['recall'][1]-metrics_1C['recall'][1])*100:+.2f}%")
    print(f"{'F1 NO':<20} {metrics_1C['f1'][0]*100:.2f}%    {metrics_3C['f1'][0]*100:.2f}%    {(metrics_3C['f1'][0]-metrics_1C['f1'][0])*100:+.2f}%")
    print(f"{'F1 LE':<20} {metrics_1C['f1'][1]*100:.2f}%    {metrics_3C['f1'][1]*100:.2f}%    {(metrics_3C['f1'][1]-metrics_1C['f1'][1])*100:+.2f}%")
    print(f"{'Specificity NO':<20} {metrics_1C['specificity'][0]*100:.2f}%    {metrics_3C['specificity'][0]*100:.2f}%    {(metrics_3C['specificity'][0]-metrics_1C['specificity'][0])*100:+.2f}%")
    print(f"{'Specificity LE':<20} {metrics_1C['specificity'][1]*100:.2f}%    {metrics_3C['specificity'][1]*100:.2f}%    {(metrics_3C['specificity'][1]-metrics_1C['specificity'][1])*100:+.2f}%")
    print("="*60)

# =============================================
# 6. MAIN
# =============================================

def main():
    print("="*60)
    print("📊 VISUALISASI PERBANDINGAN 1C vs 3C")
    print("="*60)
    
    plot_confusion_matrices()
    plot_metrics_comparison()
    plot_specificity()
    print_summary()
    
    print(f"\n✅ Semua visualisasi disimpan di: {OUTPUT_DIR}")
    print("="*60)

if __name__ == "__main__":
    main()

📊 VISUALISASI PERBANDINGAN 1C vs 3C
✅ Confusion matrix saved to /Volumes/Extreme SSD/mcu_quake_output_replikasi_demo/indonesia_domain/visualisasi_perbandingan/confusion_matrix_1C_vs_3C.png
✅ Metrics comparison saved to /Volumes/Extreme SSD/mcu_quake_output_replikasi_demo/indonesia_domain/visualisasi_perbandingan/metrics_comparison.png
✅ Metrics CSV saved to /Volumes/Extreme SSD/mcu_quake_output_replikasi_demo/indonesia_domain/visualisasi_perbandingan/metrics_comparison.csv
✅ Specificity comparison saved to /Volumes/Extreme SSD/mcu_quake_output_replikasi_demo/indonesia_domain/visualisasi_perbandingan/specificity_comparison.png

📊 RINGKASAN METRIK MCU-QUAKE INDONESIA
Metric               1C           3C           Delta     
------------------------------------------------------------
Accuracy             69.47%    69.26%    -0.21%
Precision NO         75.53%    73.96%    -1.58%
Precision LE         65.74%    66.10%    +0.37%
Recall NO            57.60%    59.46%    +1.85%
Recall LE      